# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize model (if not done in a previous cell)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?"),
]

# Compute similarities
print("==================================================")
print("          SEMANTIC SIMILARITY RESULTS             ")
print("==================================================")

for doc1, doc2 in sentence_pairs:  
    # Encode both sentences into vectors
    emb1 = model.encode([doc1])
    emb2 = model.encode([doc2])

    # Calculate cosine similarity
    similarity = cosine_similarity(emb1, emb2)[0][0]

    print(f"Sentence 1: '{doc1}'")
    print(f"Sentence 2: '{doc2}'")
    print(f"Similarity Score: {similarity:.4f}")
    print("-" * 50)

#YOUR CODE HERE

W1027 17:16:31.185000 56825 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

          SEMANTIC SIMILARITY RESULTS             
Sentence 1: 'A dog is playing in the park.'
Sentence 2: 'A dog is running in a field.'
Similarity Score: 0.5220
--------------------------------------------------
Sentence 1: 'I love pizza.'
Sentence 2: 'I enjoy ice cream.'
Similarity Score: 0.5281
--------------------------------------------------
Sentence 1: 'What is AI?'
Sentence 2: 'How does a computer learn?'
Similarity Score: 0.3194
--------------------------------------------------


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


Question 1

Answer:
The pair "I love pizza." vs. "I enjoy ice cream." is the most semantically similar with a score of 0.5281 (closely followed by the dog pair at 0.5220).
Why:
Both sentences express an identical sentiment (positive appreciation/enjoyment) toward a similar category of items (food/desserts). The embedding model recognizes that "love" and "enjoy" are close synonyms in context, and that "pizza" and "ice cream" share semantic space as popular foods.

Question 2

Answer: It can fail in multiple cases like

1. Where two cases are almost same but only differ with a single negative word. for example. "I am happy" and "I am not happy" although they are completely different mean they will produce a very good similarity score. 
2. Also, i guess the model canot understand sarcastic sentences. 

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Sentence pairs
sentence_pairs = [("I am happy", "I am not happy")]

for doc1, doc2 in sentence_pairs:
    # 1. Encode both sentences into vectors
    emb1 = model.encode([doc1])
    emb2 = model.encode([doc2])

    # 2. Calculate cosine similarity
    similarity = cosine_similarity(emb1, emb2)

    # 3. Print the results
    print(f"Sentence 1: '{doc1}'")
    print(f"Sentence 2: '{doc2}'")
    print(f"Similarity Score: {similarity[0][0]:.4f}")


Sentence 1: 'I am happy'
Sentence 2: 'I am not happy'
Similarity Score: 0.7201


We can see although they are differtent they shows 0.72 similarity score.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [4]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode the list of documents into embedding vectors
document_embeddings = model.encode(documents)

# Print the shape to confirm it created a matrix
print(f"Embeddings shape: {document_embeddings.shape}")

#YOUR CODE HERE

Embeddings shape: (6, 384)


In [5]:
# Perform KMeans clustering

#YOUR CODE HERE
import os

# Fix for macOS kernel crash issue with OpenMP/KMeans
os.environ["OMP_NUM_THREADS"] = "1"

from sklearn.cluster import KMeans

# 1. Initialize KMeans with 3 clusters
num_clusters = 3
kmeans = KMeans(n_clusters=num_clusters, n_init="auto", random_state=0)

# 2. Fit the model on our document embeddings
kmeans.fit(document_embeddings)

# 3. Get the cluster assignments for each sentence
labels = kmeans.labels_

# 4. Group and display the sentences by their assigned cluster
for i in range(num_clusters):
    print(f"\n================ CLUSTER {i} ================")
    for idx, label in enumerate(labels):
        if label == i:  # Fixed typo to use the loop variable 'i'
            print(f"- {documents[idx]}")


================ CLUSTER 0 ================
- How do I bake a chocolate cake?
- How do I change a flat tire on a car?
- How do I fix a leaky faucet?

================ CLUSTER 1 ================
- What is the capital of France?
- What is the distance between Earth and Mars?

================ CLUSTER 2 ================
- What is the best way to learn Python?


In [6]:
# Print cluster assignments

#YOUR CODE HERE
# Print the raw array of cluster assignments
print("Raw cluster labels for each document:")
print(labels)


Raw cluster labels for each document:
[1 0 1 0 2 0]


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice


1. How many clusters make the most sense? Why?

3 clusters make the most sense for this specific 6-sentence dataset.
The sentences naturally partition into three distinct operational domains: factual knowledge requests ("What is..."), practical hands-on repairs or recipes ("How do I..."), and computer programming instruction. Forcing the data into 2 clusters would blend unrelated topics together, while 4 or more clusters would unnecessarily fragment tightly related themes.

2. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?

Yes, the groupings are highly meaningful and cluster well based on intent:
• Cluster 0: Practical "How-To" Guides & DIY Tasks
	• Theme: Instructions for physical tasks (baking a cake, changing a tire, fixing a faucet).
• Cluster 1: Factual Science & Geography Knowledge
	• Theme: Direct geographic and astronomical facts (capital of France, distance to Mars).
• Cluster 2: Coding & Technical Education
	• Theme: Learning digital skills (best way to master Python coding).

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [7]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings

doc_embeddings = model.encode(documents)

# Verify the matrix shape
print(f"Generated embeddings shape: {doc_embeddings.shape}")


Generated embeddings shape: (10, 384)


In [12]:
# Create the search function
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # YOUR CODE HERE
    # 1. Encode the user query into a semantic vector
    query_embedding = model.encode(query)
    
    # 2. Compute cosine similarity between the query and all document embeddings
    dot_product = np.dot(doc_embeddings, query_embedding)
    doc_norms = np.linalg.norm(doc_embeddings, axis=1)
    query_norm = np.linalg.norm(query_embedding)
    similarities = dot_product / (doc_norms * query_norm)
    
    # 3. Sort similarities in descending order and get the top N indices
    top_indices = np.argsort(similarities)[::-1][:top_n]
    
    # 4. Return the corresponding top N documents along with their scores (optional but helpful)
    results = [documents[i] for i in top_indices]
    
    return results

In [13]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

['What is quantum computing?',
 'What is the best way to learn Python?',
 'How do I build a mobile app?',
 'How do I set up a local server?',
 'What are the best travel destinations in Europe?']

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset


1. What are the top-ranked results for the given queries?

For the test query "Explain programming languages.", the top 5 returned results from your dataset are:
1. 'What is quantum computing?'
2. 'What is the best way to learn Python?'
3. 'How do I build a mobile app?'
4. 'How do I set up a local server?'
5. 'What are the best travel destinations in Europe?'
(Note: "What is the best way to learn Python?" and "How do I build a mobile app?" are semantically the closest to "programming languages" out of the limited 10-document dataset available).

2. How can you improve the ranking explanation for users?

To make the results transparent and intuitive for an end-user, you can implement the following enhancements:
• Display Similarity Scores: Instead of just returning raw strings, print out the exact percentage match or cosine score (e.g., Match: 78%) next to each document so the user understands the model's confidence.
• Highlight Keyword/Concept Triggers: Use text highlighting to show which words or contextual concepts in the document most heavily influenced the dense vector match.
• Set a Relevancy Threshold: Filter out results below a certain similarity threshold (e.g., < 0.30) so that totally irrelevant matches (like the "travel destinations" document) aren't shown just to fill up the top_n count.